In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 2: Values, the Bellman equation, and planning with a model

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 2: the tour gave the whole picture
    - this session fills in the *value*: how good is each state?
    - the next session learns values from experience, when the model is unknown
- **This session's goal**
    - compute values and a good policy on Frozen Lake when the model is known
- **By the end you can**
    - define the state value $\statevalfun_\pi$ and write the Bellman equation
    - run value iteration and policy iteration on a small grid
    - explain why a policy needs a one-step lookahead or action values
    - say what changes when the model is not known


# Value-based methods: overview

*Value-based* methods come in model-based and model-free versions (this session: model-based; the next: model-free).

They revolve around the idea of
- creating a *State Value Function* $\statevalfun_\pi(\state)$ 
- to assign a value to each state $\state \in \States$
$$
\statevalfun_{\pi}: \States \to \Reals
$$

$\statevalfun_\pi(\state)$ is the expectation 
$$
\E_\pi ( \rlret_\tt  | \stateseq_\tt =  \state )
$$
the expected return from state $\state$ when following $\pi$

**Frozen Lake in this session**

- the grid has $16$ squares, numbered $0$ to $15$ row by row from the top left (**S** is square $0$, **G** is square $15$)
- four actions: left, down, right, up; **non-slippery** here: every move goes where intended (a slippery version comes later)
- reward: $+1$ when a move enters **G**, $0$ otherwise; the episode ends at **G** or at a hole **H**
- the model: $\transp(\state' \mid \state, \act)$ is the chance of landing in square $\state'$ (here $0$ or $1$), and $\rew(\state, \act, \state')$ is the reward for that move; a move into the outer wall leaves the agent where it is
- discount $\disc = 0.9$
- terminal squares (**G** and the holes) have value $0$: the $+1$ is earned by the move into **G**, not by being there


We usually represent (and implement) $\statevalfun_\pi(\state)$ as a table.
- for the case of finite number of states

For Frozen Lake the *uninitialized* table (all values $0$) looks like:

<br>
<table>
    <center><strong>Value function (uninitialized) as a table</strong></center>
<tr>
    <td><img src="images/Frozen_Lake_env.png"></td>
    <td><img src="images/value_table_zero.png"></td>
</tr>
</table>

Each cell
- corresponds to a state
- contains a real-valued expected value of return-to-go from that state
    - colors correspond to magnitude: darker green is higher, pale is $0$
    
In Frozen Lake
- there is a single reward of +1
    - hence: expected returns in each cell are in the range $[0,1]$
- earned by the move into the terminal state (G)

Given the optimal value function $\statevalfun_*$, an optimal deterministic policy looks one step ahead

$$
\pi_*(\state) = \argmax{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc\,\statevalfun_*(\state')\bigr)
$$

- From state $\state$
- try each action $\act$
- use the model $\transp$ to weigh each possible next state $\state'$
- choose the action with the best reward plus discounted value of what follows

Note: the $\argmax{}$ results in a *deterministic* policy.

Without the model this lookahead is impossible; action values (later in this session) make the last step model-free, and the next session learns them from experience.


Using a value-based method is practical only if we can
discover the state value function $\statevalfun$, which is initially unknown.

Most methods are iterative in nature and create a sequence of increasingly accurate approximations of $\statevalfun$ 
$$
\statevalfun_{\pi,0} \ldots \statevalfun_{\pi,k} \ldots
$$

In the limit 
$$
\lim_{k \to \infty}  \statevalfun_{\pi,k} = \statevalfun_\pi 
$$

As we obtain an improved approximation $\statevalfun_{\pi, k+1}$ 
- we may reflect this improved knowledge by updating the policy

Thus, we periodically improve the policy based on improved approximations of $\statevalfun_\pi$

This results in a sequence of increasingly accurate approximations of the policy $\pi$

$$
\pi_0, \ldots, \pi_p, \ldots
$$
which hopefully converges to $\pi_*$.

Here is the final value (non-slippery, discount $0.9$) of
- $\statevalfun_*(\state)$ 
- and  $\pi_*(\state)$, shown as arrows

<table>
    <center><strong>Value function (converged) as a table</strong></center>
<tr>
    <td><img src="images/value_iteration_final.png"></td>
</tr>
</table>

Where two actions tie (squares $0$ and $9$), one arrow is shown.


We will show two broad classes of Value-based methods
- Model-based
- Model-free

<br>

    Value-Based Methods
    ├── Model-Based
    │   └── DP Methods
    │       ├── Value Iteration
    │       └── Policy Iteration
    │
    └── Model-Free
        ├── Monte Carlo (MC)
        ├── Temporal Difference (TD)
        └── Q-Learning



## Backups and the Bellman equation

A common characteristic of both Model-based and Model-free Value methods
- the same idea: the Bellman equation (model-based methods apply it exactly, as Dynamic Programming; model-free methods sample it).

In Dynamic Programming
- there is a *Bellman update* equation
- which relates the value of a state $\state$
- to the values of each of the potential successor states $\state'$
    - which the environment may return in response to the Agent taking an action in state $\state$
    
Information flowing from successor to predecessor state is called a *backup*


The Bellman equation says that the value of a state is an average over what can happen next
$$
\statevalfun_\pi(\state) = \sum_{\act} \pi(\act \mid \state) \sum_{\state'} \transp(\state' \mid \state, \act)\bigl(\rew(\state,\act,\state') + \disc\,\statevalfun_\pi(\state')\bigr)
$$

- the immediate reward $\rew(\state, \act, \state')$
    - received by taking action $\act$ in state $\state$ and transitioning to state $\state'$
- plus the discounted (by $\disc$) value $\statevalfun_\pi(\state')$ of the successor state
- averaged over the actions the policy may choose and the successors the environment may return
- why: the return satisfies $\rlret_\tt = \rewseq_{\tt+1} + \disc\,\rlret_{\tt+1}$, so the expected return from $\state$ is the expected first reward plus $\disc$ times the expected return from wherever we land
- on the real, *slippery* lake a move reaches several squares (one third each), so $\transp$ has several nonzero terms and the sum is a real average; on today's non-slippery lake it has a single term


## Model-based vs Model-free methods

The main difference between Model-based and Model-free methods
- a Model-based method does *not need to actively gather* experience
    - if the model is completely known
    - all necessary information is available without actively interacting with the Environment
        - $\transp(\state' \mid \state, \act)$ is known $\forall \state, \state' \in \States, \act \in \Actions$
       
- a Model-free method *must* actively interact with the Environment
    - gathers a *sample* from $\transp(\state' \mid \state, \act)$ for each $\state, \act$ pair in the episode
    - through multiple episodes
        - the *effect* of the true $\transp(\state' \mid \state, \act)$ is derived

- here *model-based* means the model is known and used for planning; the tour's "learning a model" (estimating it from experience) is a third idea


| Method           | Category       | Model-based? | Update Equation &nbsp;  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;                                                        | Key Characteristics                           | On-/Off-policy        | Convergence Target          |
|:-----------------|:--------------|:-------------|:--------------------------------------------------------------------------|:-----------------------------------------------|:---------------------|:---------------------------|
| Value Iteration | Value-based | Yes | $ \statevalfun(\state) \leftarrow \max{\act} \sum_{\state'} \transp(\state' \mid \state, \act)\bigl[\rew(\state,\act,\state') + \disc\,\statevalfun(\state')\bigr] $ | Uses full model; synchronous updates | N/A | $ \statevalfun_* $, $ \pi_* $ |
| Policy Iteration | Value-based (DP) | Yes | Eval: $ \statevalfun_\pi(\state) \leftarrow \sum_{\state'} \transp(\state' \mid \state, \pi(\state))\bigl[\rew(\state,\pi(\state),\state') + \disc\,\statevalfun_\pi(\state')\bigr] $<br>Improve: $ \pi(\state) \leftarrow \argmax{\act} \actvalfun_\pi(\state,\act) $ | Alternates evaluation/policy update | N/A | $ \statevalfun_* $, $ \pi_* $ |
| TD(0) | Value-based | No | $ \statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \alpha\bigl[\rewseq_{\tt+1} + \disc\,\statevalfun(\stateseq_{\tt+1}) - \statevalfun(\stateseq_\tt)\bigr] $ | Incremental via samples, bootstraps | On-policy | $ \statevalfun_\pi $ |
| Q-learning | Value-based | No | $ \actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha\bigl[\rewseq_{\tt+1} + \disc \max{\act} \actvalfun(\stateseq_{\tt+1}, \act) - \actvalfun(\stateseq_\tt, \actseq_\tt)\bigr] $ | Off-policy, learns optimal $ \actvalfun_* $ | Off-policy | $ \actvalfun_* $, $ \pi_* $ |
| Monte Carlo | Value-based | No | $ \statevalfun(\stateseq_\tt) \leftarrow \statevalfun(\stateseq_\tt) + \alpha\bigl[\rlret_\tt - \statevalfun(\stateseq_\tt)\bigr] $, where $ \rlret_\tt $ = return from step $ \tt $ of the episode | No bootstrapping; full episodes | On-policy | $ \statevalfun_\pi $ |
| SARSA | Value-based | No | $ \actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha\bigl[\rewseq_{\tt+1} + \disc\,\actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \actvalfun(\stateseq_\tt, \actseq_\tt)\bigr] $ | On-policy; backup uses sampled $ \actseq_{\tt+1} $ | On-policy | $ \actvalfun_\pi $ |


In [2]:
print("Done")

Done
